# 📊 Goldmine ML Strategy - Data Exploration

**Objective:** Load, validate, and explore multi-timeframe market data for XAUUSDm

**Data:** M1, M3, M5, H1 OHLCV from January 2024 to July 2026 (18 months)

**Tasks:**
1. Load all timeframe data
2. Data quality validation
3. Exploratory analysis and visualization
4. Market regime identification
5. Summary statistics and report
6. Save cleaned data

---

## 🔧 Setup & Imports

In [1]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import os
import json

warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (15, 8)
%matplotlib inline

print('✅ Libraries imported successfully!')
print(f'📅 Notebook executed: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

✅ Libraries imported successfully!
📅 Notebook executed: 2026-07-04 19:01:08


## 📂 1. Data Loading

Load all four timeframes: M1, M3, M5, H1

In [2]:
# Data file paths
data_files = {
    'M1': '../data/raw/XAUUSDm_M1_202401012305_202607031457.csv',
    'M3': '../data/raw/XAUUSDm_M3_202401012303_202607031457.csv',
    'M5': '../data/raw/XAUUSDm_M5_202401012305_202607031500.csv',
    'H1': '../data/raw/XAUUSDm_H1_202401012300_202607031500.csv'
}

# Check if files exist
print("🔍 Checking data files...\n")
for tf, path in data_files.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"✅ {tf}: {path.split('/')[-1]} ({size_mb:.2f} MB)")
    else:
        print(f"❌ {tf}: File not found - {path}")

🔍 Checking data files...

✅ M1: XAUUSDm_M1_202401012305_202607031457.csv (56.28 MB)
✅ M3: XAUUSDm_M3_202401012303_202607031457.csv (18.93 MB)
✅ M5: XAUUSDm_M5_202401012305_202607031500.csv (11.39 MB)
✅ H1: XAUUSDm_H1_202401012300_202607031500.csv (0.97 MB)


In [3]:
# Load data function
def load_timeframe_data(filepath, timeframe):
    """
    Load and parse market data from CSV
    
    Parameters:
    - filepath: str, path to CSV file
    - timeframe: str, timeframe identifier (M1, M3, M5, H1)
    
    Returns:
    - DataFrame with parsed datetime and timeframe column
    """
    print(f"\n📥 Loading {timeframe} data...")
    
    # Read CSV
    df = pd.read_csv(filepath)
    
    # Parse datetime (try different column names)
    datetime_cols = ['timestamp', 'time', 'datetime', 'Time', 'Timestamp']
    datetime_col = None
    
    for col in datetime_cols:
        if col in df.columns:
            datetime_col = col
            break
    
    if datetime_col:
        df['timestamp'] = pd.to_datetime(df[datetime_col])
        if datetime_col != 'timestamp':
            df = df.drop(columns=[datetime_col])
    else:
        raise ValueError(f"No datetime column found in {filepath}")
    
    # Standardize column names (lowercase)
    df.columns = [col.lower() for col in df.columns]
    
    # Add timeframe identifier
    df['timeframe'] = timeframe
    
    # Sort by timestamp
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    print(f"   Shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")
    print(f"   Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
    print(f"   Duration: {(df['timestamp'].max() - df['timestamp'].min()).days} days")
    
    return df

In [4]:
# Load all timeframes
data = {}

for tf, path in data_files.items():
    try:
        data[tf] = load_timeframe_data(path, tf)
        print(f"✅ {tf} loaded successfully\n")
    except Exception as e:
        print(f"❌ Error loading {tf}: {e}\n")

print("\n" + "="*60)
print("📊 DATA LOADING SUMMARY")
print("="*60)
for tf, df in data.items():
    print(f"{tf}: {len(df):,} rows | {df['timestamp'].min().date()} to {df['timestamp'].max().date()}")


📥 Loading M1 data...
❌ Error loading M1: No datetime column found in ../data/raw/XAUUSDm_M1_202401012305_202607031457.csv


📥 Loading M3 data...
❌ Error loading M3: No datetime column found in ../data/raw/XAUUSDm_M3_202401012303_202607031457.csv


📥 Loading M5 data...
❌ Error loading M5: No datetime column found in ../data/raw/XAUUSDm_M5_202401012305_202607031500.csv


📥 Loading H1 data...
❌ Error loading H1: No datetime column found in ../data/raw/XAUUSDm_H1_202401012300_202607031500.csv


📊 DATA LOADING SUMMARY


## 🔍 2. Data Quality Validation

Check for:
- Missing values
- Duplicate timestamps
- Invalid OHLC relationships
- Unrealistic price jumps
- Volume anomalies

In [8]:
def validate_data_quality(df, timeframe):
    """
    Comprehensive data quality validation
    """
    print(f"\n🔍 VALIDATING {timeframe} DATA")
    print("="*50)
    
    issues = []
    
    # 1. Missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(f"⚠️  Missing values found:")
        print(missing[missing > 0])
        issues.append("missing_values")
    else:
        print("✅ No missing values")
    
    # 2. Duplicate timestamps
    duplicates = df['timestamp'].duplicated().sum()
    if duplicates > 0:
        print(f"⚠️  {duplicates} duplicate timestamps found")
        issues.append("duplicates")
    else:
        print("✅ No duplicate timestamps")
    
    # 3. OHLC relationship validation
    invalid_ohlc = (
        (df['low'] > df['open']) | 
        (df['low'] > df['close']) | 
        (df['high'] < df['open']) | 
        (df['high'] < df['close']) |
        (df['low'] > df['high'])
    ).sum()
    
    if invalid_ohlc > 0:
        print(f"⚠️  {invalid_ohlc} invalid OHLC relationships")
        issues.append("invalid_ohlc")
    else:
        print("✅ All OHLC relationships valid")
    
    # 4. Unrealistic price jumps (>10% in one candle)
    df['price_change_pct'] = df['close'].pct_change().abs() * 100
    large_jumps = (df['price_change_pct'] > 10).sum()
    if large_jumps > 0:
        print(f"⚠️  {large_jumps} price jumps >10%")
        print(f"   Max jump: {df['price_change_pct'].max():.2f}%")
        issues.append("large_jumps")
    else:
        print("✅ No unrealistic price jumps")
    
    # 5. Volume validation
    if 'volume' in df.columns:
        zero_volume = (df['volume'] == 0).sum()
        negative_volume = (df['volume'] < 0).sum()
        
        if zero_volume > 0:
            print(f"⚠️  {zero_volume} candles with zero volume ({zero_volume/len(df)*100:.2f}%)")
        if negative_volume > 0:
            print(f"❌ {negative_volume} candles with negative volume")
            issues.append("negative_volume")
        if zero_volume == 0 and negative_volume == 0:
            print("✅ Volume data looks good")
    
    # 6. Time gaps
    df['time_diff'] = df['timestamp'].diff()
    expected_diff = {
        'M1': pd.Timedelta('1 minute'),
        'M3': pd.Timedelta('3 minutes'),
        'M5': pd.Timedelta('5 minutes'),
        'H1': pd.Timedelta('1 hour')
    }
    
    if timeframe in expected_diff:
        gaps = (df['time_diff'] > expected_diff[timeframe] * 3).sum()  # More than 3x expected
        if gaps > 0:
            print(f"⚠️  {gaps} time gaps detected (>3x expected interval)")
            issues.append("time_gaps")
        else:
            print("✅ No significant time gaps")
    
    print("\n" + "="*50)
    if len(issues) == 0:
        print(f"🎉 {timeframe}: DATA QUALITY EXCELLENT")
    else:
        print(f"⚠️  {timeframe}: {len(issues)} issue(s) found: {', '.join(issues)}")
    
    return issues

In [6]:
# Validate all timeframes
quality_report = {}

for tf, df in data.items():
    quality_report[tf] = validate_data_quality(df, tf)

## 📊 3. Exploratory Data Analysis

### 3.1 Summary Statistics

In [7]:
# Summary statistics for M5 (primary timeframe)
print("📊 M5 TIMEFRAME - SUMMARY STATISTICS")
print("="*60)

m5 = data['M5']

# Price statistics
print("\n💰 PRICE STATISTICS:")
print(f"   Close Price Range: ${m5['close'].min():.2f} - ${m5['close'].max():.2f}")
print(f"   Mean Close: ${m5['close'].mean():.2f}")
print(f"   Median Close: ${m5['close'].median():.2f}")
print(f"   Std Dev: ${m5['close'].std():.2f}")

# Volatility
m5['candle_range'] = m5['high'] - m5['low']
print(f"\n📈 VOLATILITY:")
print(f"   Avg Candle Range: ${m5['candle_range'].mean():.2f}")
print(f"   Max Candle Range: ${m5['candle_range'].max():.2f}")
print(f"   90th Percentile: ${m5['candle_range'].quantile(0.90):.2f}")

# Volume
if 'volume' in m5.columns:
    print(f"\n📊 VOLUME:")
    print(f"   Total Volume: {m5['volume'].sum():,.0f}")
    print(f"   Avg Volume: {m5['volume'].mean():,.0f}")
    print(f"   Max Volume: {m5['volume'].max():,.0f}")

# General info
print(f"\n🔢 DATA OVERVIEW:")
print(f"   Total Candles: {len(m5):,}")
print(f"   Duration: {(m5['timestamp'].max() - m5['timestamp'].min()).days} days")
print(f"   Start: {m5['timestamp'].min()}")
print(f"   End: {m5['timestamp'].max()}")

📊 M5 TIMEFRAME - SUMMARY STATISTICS


KeyError: 'M5'

### 3.2 Price Visualization

In [ ]:
# Plot closing prices for all timeframes
fig, axes = plt.subplots(4, 1, figsize=(16, 12))
fig.suptitle('XAUUSDm Closing Prices - All Timeframes', fontsize=16, fontweight='bold')

for idx, (tf, df) in enumerate(data.items()):
    axes[idx].plot(df['timestamp'], df['close'], linewidth=0.8, alpha=0.8)
    axes[idx].set_title(f'{tf} Timeframe', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('Price ($)', fontsize=10)
    axes[idx].grid(True, alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=45)
    
    # Add mean line
    axes[idx].axhline(y=df['close'].mean(), color='r', linestyle='--', 
                       linewidth=1, alpha=0.5, label=f'Mean: ${df["close"].mean():.2f}')
    axes[idx].legend(loc='upper left')

axes[-1].set_xlabel('Date', fontsize=10)
plt.tight_layout()
plt.show()

print("✅ Price charts generated")